In [1]:
!pip install transformers datasets accelerate scikit-learn pandas openpyxl -q

In [2]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
file_path = "/content/5K_HateSpeechAllGender.xlsx"

# Read without header because your file has no proper header row
df = pd.read_excel(file_path, header=None)

print("Original shape:", df.shape)
print(df.head())

Original shape: (5320, 24)
                                                  0     1   \
0  S K  யாருடா நீ எனக்கே என்ன பாக்கனும் போல இருக்...  HATE   
1  டேய் உன்னோட சின்ன தங்கச்சி கொத்தும் கொலையுமா ந...  HATE   
2  பிரபல நாட்டாமை சுண்ணி ஊம்ப காசு அதிகம் வேணுமா?...  HATE   
3           அது ஒரு பொறம்போக்கு சிந்தனை கொண்ட ஜந்து.  HATE   
4   என்னோட சுன்னிய சப்பிட்டு தான் உன் குண்டியை கா...  HATE   

                    2         3   \
0     Hate against Men  IMPLICIT   
1  Hate against LGBTQ+  EXPLICIT   
2  Hate against LGBTQ+  EXPLICIT   
3   No Target- Profane  EXPLICIT   
4  Hate against LGBTQ+  EXPLICIT   

                                              4           5    6    7    8   \
0  ImplicitMen-Undermining / Mocking Masculinity  Religional  NaN  NaN  NaN   
1                 ExplicitLGBTQ-Homophobic Slurs         NaN  NaN  NaN  NaN   
2                 ExplicitLGBTQ-Homophobic Slurs         Nil   No  NaN  NaN   
3                                            Nil         Nil   No  

In [5]:
# Your first 6 columns are the useful ones
df = df.iloc[:, :6]

df.columns = [
    "Text",
    "Level 1 - Class",
    "Level 2 - Target",
    "Level 3 - Category",
    "Level 4 - Subcategory",
    "Level 5 - Bias"
]

print(df.shape)
df.head()

(5320, 6)


,Text,Level 1 - Class,Level 2 - Target,Level 3 - Category,Level 4 - Subcategory,Level 5 - Bias
0,S K யாருடா நீ எனக்கே என்ன பாக்கனும் போல இருக்...,HATE,Hate against Men,IMPLICIT,ImplicitMen-Undermining / Mocking Masculinity,Religional
1,டேய் உன்னோட சின்ன தங்கச்சி கொத்தும் கொலையுமா ந...,HATE,Hate against LGBTQ+,EXPLICIT,ExplicitLGBTQ-Homophobic Slurs,NaN
2,பிரபல நாட்டாமை சுண்ணி ஊம்ப காசு அதிகம் வேணுமா?...,HATE,Hate against LGBTQ+,EXPLICIT,ExplicitLGBTQ-Homophobic Slurs,Nil
3,அது ஒரு பொறம்போக்கு சிந்தனை கொண்ட ஜந்து.,HATE,No Target- Profane,EXPLICIT,Nil,Nil
4,என்னோட சுன்னிய சப்பிட்டு தான் உன் குண்டியை கா...,HATE,Hate against LGBTQ+,EXPLICIT,ExplicitLGBTQ-Homophobic Slurs,NaN


In [6]:
TEXT_COL = "Text"

LEVEL1_COL = "Level 1 - Class"
LEVEL2_COL = "Level 2 - Target"
LEVEL3_COL = "Level 3 - Category"
LEVEL4_COL = "Level 4 - Subcategory"
LEVEL5_COL = "Level 5 - Bias"

label_cols = [
    LEVEL1_COL,
    LEVEL2_COL,
    LEVEL3_COL,
    LEVEL4_COL,
    LEVEL5_COL
]

In [8]:
df = df.copy()

df[TEXT_COL] = df[TEXT_COL].astype(str).fillna("")

for col in label_cols:
    df[col] = df[col].fillna("Nil")
    df[col] = df[col].astype(str)
    df[col] = df[col].str.strip()
    df[col] = df[col].replace({
        "": "Nil",
        "nan": "Nil",
        "NaN": "Nil",
        "None": "Nil"
    })

df = df[df[TEXT_COL].str.strip() != ""].reset_index(drop=True)

print(df.shape)
df.head()

(5320, 6)


,Text,Level 1 - Class,Level 2 - Target,Level 3 - Category,Level 4 - Subcategory,Level 5 - Bias
0,S K யாருடா நீ எனக்கே என்ன பாக்கனும் போல இருக்...,HATE,Hate against Men,IMPLICIT,ImplicitMen-Undermining / Mocking Masculinity,Religional
1,டேய் உன்னோட சின்ன தங்கச்சி கொத்தும் கொலையுமா ந...,HATE,Hate against LGBTQ+,EXPLICIT,ExplicitLGBTQ-Homophobic Slurs,Nil
2,பிரபல நாட்டாமை சுண்ணி ஊம்ப காசு அதிகம் வேணுமா?...,HATE,Hate against LGBTQ+,EXPLICIT,ExplicitLGBTQ-Homophobic Slurs,Nil
3,அது ஒரு பொறம்போக்கு சிந்தனை கொண்ட ஜந்து.,HATE,No Target- Profane,EXPLICIT,Nil,Nil
4,என்னோட சுன்னிய சப்பிட்டு தான் உன் குண்டியை கா...,HATE,Hate against LGBTQ+,EXPLICIT,ExplicitLGBTQ-Homophobic Slurs,Nil


In [9]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}
num_labels = {}

for col in label_cols:
    le = LabelEncoder()
    df[col + "_enc"] = le.fit_transform(df[col])
    label_encoders[col] = le
    num_labels[col] = len(le.classes_)

    print(f"\n{col}")
    print("Classes:", list(le.classes_))
    print("Number of classes:", num_labels[col])

enc_label_cols = [col + "_enc" for col in label_cols]

print("Encoded columns created:")
print(enc_label_cols)

print(df.columns.tolist())


Level 1 - Class
Classes: ['HATE', 'Level1 - CLASS', 'NON-HATE', 'Nil']
Number of classes: 4

Level 2 - Target
Classes: ['Hate against LGBTQ+', 'Hate against Men', 'Hate against Women', 'Level 2 - TARGET', 'Men- no profane', 'Nil', 'No Target- Profane']
Number of classes: 7

Level 3 - Category
Classes: ['EXPLICIT', 'IMPLICIT', 'Level 3 - CATEGORIES', 'Nil']
Number of classes: 4

Level 4 - Subcategory
Classes: ['ExplicitLGBTQ-Homophobic Slurs', 'ExplicitLGBTQ-Physical / Sexual Threats', 'ExplicitLGBTQ-Transphobic Attacks', 'ExplicitLGBTQ-Verbal Abusive', 'ExplicitMen-Masculinity Attacks', 'ExplicitMen-Verbal Abusive', 'ExplicitWomen-Sexual Objectification', 'ExplicitWomen-Sexual Threat', 'ExplicitWomen-Verbal Abusive', 'ImplicitLGBTQ-Character assassination', 'ImplicitLGBTQ-Exclusion', 'ImplicitLGBTQ-Mocking Identity', 'ImplicitLGBTQ-Moral policing', 'ImplicitLGBTQ-Pathologizing', 'ImplicitLGBTQ-Slut-shaming', 'ImplicitLGBTQ-Victim Blaming', 'ImplicitMen-Character assassination', 'Impli

In [10]:
print(df.columns.tolist())

['Text', 'Level 1 - Class', 'Level 2 - Target', 'Level 3 - Category', 'Level 4 - Subcategory', 'Level 5 - Bias', 'Level 1 - Class_enc', 'Level 2 - Target_enc', 'Level 3 - Category_enc', 'Level 4 - Subcategory_enc', 'Level 5 - Bias_enc']


In [11]:
df = df.copy()

df[TEXT_COL] = df[TEXT_COL].astype(str).fillna("")

for col in label_cols:
    df[col] = df[col].fillna("Nil")
    df[col] = df[col].astype(str)
    df[col] = df[col].str.strip()
    df[col] = df[col].replace({
        "": "Nil",
        "nan": "Nil",
        "NaN": "Nil",
        "None": "Nil"
    })

df = df[df[TEXT_COL].str.strip() != ""].reset_index(drop=True)

print(df.shape)
df[label_cols].head()

(5320, 11)


,Level 1 - Class,Level 2 - Target,Level 3 - Category,Level 4 - Subcategory,Level 5 - Bias
0,HATE,Hate against Men,IMPLICIT,ImplicitMen-Undermining / Mocking Masculinity,Religional
1,HATE,Hate against LGBTQ+,EXPLICIT,ExplicitLGBTQ-Homophobic Slurs,Nil
2,HATE,Hate against LGBTQ+,EXPLICIT,ExplicitLGBTQ-Homophobic Slurs,Nil
3,HATE,No Target- Profane,EXPLICIT,Nil,Nil
4,HATE,Hate against LGBTQ+,EXPLICIT,ExplicitLGBTQ-Homophobic Slurs,Nil


In [12]:
for col in label_cols:
    print("\n", "="*60)
    print(col)
    print(df[col].value_counts())


Level 1 - Class
Level 1 - Class
HATE              2789
NON-HATE          2519
Nil                 11
Level1 - CLASS       1
Name: count, dtype: int64

Level 2 - Target
Level 2 - Target
Nil                    2568
Hate against Women     1272
Hate against Men        774
No Target- Profane      512
Hate against LGBTQ+     183
Men- no profane          10
Level 2 - TARGET          1
Name: count, dtype: int64

Level 3 - Category
Level 3 - Category
Nil                     2542
IMPLICIT                1578
EXPLICIT                1199
Level 3 - CATEGORIES       1
Name: count, dtype: int64

Level 4 - Subcategory
Level 4 - Subcategory
Nil                                              3095
ExplicitMen-Verbal Abusive                        408
Sarcasm                                           367
ExplicitWomen-Verbal Abusive                      302
ImplicitWomen-Character assassination             213
ImplicitWomen-Slut-shaming                        128
ImplicitWomen-Dismissal                   

In [13]:
label_encoders = {}
num_labels = {}

for col in label_cols:
    le = LabelEncoder()
    df[col + "_enc"] = le.fit_transform(df[col])
    label_encoders[col] = le
    num_labels[col] = len(le.classes_)

    print(f"\n{col}")
    print("Classes:", list(le.classes_))
    print("Number of classes:", num_labels[col])


Level 1 - Class
Classes: ['HATE', 'Level1 - CLASS', 'NON-HATE', 'Nil']
Number of classes: 4

Level 2 - Target
Classes: ['Hate against LGBTQ+', 'Hate against Men', 'Hate against Women', 'Level 2 - TARGET', 'Men- no profane', 'Nil', 'No Target- Profane']
Number of classes: 7

Level 3 - Category
Classes: ['EXPLICIT', 'IMPLICIT', 'Level 3 - CATEGORIES', 'Nil']
Number of classes: 4

Level 4 - Subcategory
Classes: ['ExplicitLGBTQ-Homophobic Slurs', 'ExplicitLGBTQ-Physical / Sexual Threats', 'ExplicitLGBTQ-Transphobic Attacks', 'ExplicitLGBTQ-Verbal Abusive', 'ExplicitMen-Masculinity Attacks', 'ExplicitMen-Verbal Abusive', 'ExplicitWomen-Sexual Objectification', 'ExplicitWomen-Sexual Threat', 'ExplicitWomen-Verbal Abusive', 'ImplicitLGBTQ-Character assassination', 'ImplicitLGBTQ-Exclusion', 'ImplicitLGBTQ-Mocking Identity', 'ImplicitLGBTQ-Moral policing', 'ImplicitLGBTQ-Pathologizing', 'ImplicitLGBTQ-Slut-shaming', 'ImplicitLGBTQ-Victim Blaming', 'ImplicitMen-Character assassination', 'Impli

In [14]:
enc_label_cols = [col + "_enc" for col in label_cols]

In [15]:
# Before splitting, filter out rows where 'Level 1 - Class' has only one instance.
# Based on previous value_counts, 'Level1 - CLASS' has only 1 occurrence.
# Filtering this out will prevent the ValueError during stratified splitting.
df_filtered = df[df[LEVEL1_COL] != 'Level1 - CLASS'].copy()

# Re-encode all label columns on df_filtered to ensure the _enc columns exist for stratification and dataset creation.
# This step is added here because df might have been reset upstream, removing encoded columns.
for col in label_cols:
    le = LabelEncoder()
    df_filtered[col + '_enc'] = le.fit_transform(df_filtered[col])
    # Update the global label_encoders and num_labels with filtered data's encoding
    label_encoders[col] = le
    num_labels[col] = len(le.classes_)

# Verify columns in df_filtered AFTER encoding
print("Columns in df_filtered AFTER encoding:", df_filtered.columns.tolist())

train_df, temp_df = train_test_split(
    df_filtered, # Use the filtered DataFrame
    test_size=0.2,
    random_state=SEED,
    stratify=df_filtered[LEVEL1_COL + "_enc"] # Stratify on the encoded column of the filtered DataFrame
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=SEED,
    stratify=temp_df[LEVEL1_COL + "_enc"]
)

# Verify columns in split dataframes
print("Columns in train_df:", train_df.columns.tolist())
print("Columns in val_df:", val_df.columns.tolist())
print("Columns in test_df:", test_df.columns.tolist())

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Columns in df_filtered AFTER encoding: ['Text', 'Level 1 - Class', 'Level 2 - Target', 'Level 3 - Category', 'Level 4 - Subcategory', 'Level 5 - Bias', 'Level 1 - Class_enc', 'Level 2 - Target_enc', 'Level 3 - Category_enc', 'Level 4 - Subcategory_enc', 'Level 5 - Bias_enc']
Columns in train_df: ['Text', 'Level 1 - Class', 'Level 2 - Target', 'Level 3 - Category', 'Level 4 - Subcategory', 'Level 5 - Bias', 'Level 1 - Class_enc', 'Level 2 - Target_enc', 'Level 3 - Category_enc', 'Level 4 - Subcategory_enc', 'Level 5 - Bias_enc']
Columns in val_df: ['Text', 'Level 1 - Class', 'Level 2 - Target', 'Level 3 - Category', 'Level 4 - Subcategory', 'Level 5 - Bias', 'Level 1 - Class_enc', 'Level 2 - Target_enc', 'Level 3 - Category_enc', 'Level 4 - Subcategory_enc', 'Level 5 - Bias_enc']
Columns in test_df: ['Text', 'Level 1 - Class', 'Level 2 - Target', 'Level 3 - Category', 'Level 4 - Subcategory', 'Level 5 - Bias', 'Level 1 - Class_enc', 'Level 2 - Target_enc', 'Level 3 - Category_enc', 'Lev

In [16]:
MODEL_NAME = "google/muril-base-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

In [17]:
class HateSpeechDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len, text_col, label_cols):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.text_col = text_col
        self.label_cols = label_cols

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        text = str(row[self.text_col])

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt"
        )

        labels = {}
        for col in self.label_cols:
            labels[col] = torch.tensor(row[col], dtype=torch.long)

        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": labels
        }

        return item

In [18]:
MAX_LEN = 128
BATCH_SIZE = 16

train_dataset = HateSpeechDataset(
    train_df,
    tokenizer,
    MAX_LEN,
    TEXT_COL,
    enc_label_cols
)

val_dataset = HateSpeechDataset(
    val_df,
    tokenizer,
    MAX_LEN,
    TEXT_COL,
    enc_label_cols
)

test_dataset = HateSpeechDataset(
    test_df,
    tokenizer,
    MAX_LEN,
    TEXT_COL,
    enc_label_cols
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [19]:
print("Missing values in train_df:\n", train_df.isnull().sum())

Missing values in train_df:
 Text                         0
Level 1 - Class              0
Level 2 - Target             0
Level 3 - Category           0
Level 4 - Subcategory        0
Level 5 - Bias               0
Level 1 - Class_enc          0
Level 2 - Target_enc         0
Level 3 - Category_enc       0
Level 4 - Subcategory_enc    0
Level 5 - Bias_enc           0
dtype: int64


In [20]:
print("Label columns in train_dataset:", train_dataset.label_cols)

Label columns in train_dataset: ['Level 1 - Class_enc', 'Level 2 - Target_enc', 'Level 3 - Category_enc', 'Level 4 - Subcategory_enc', 'Level 5 - Bias_enc']


In [21]:
class HierarchicalMuRIL(nn.Module):
    def __init__(
        self,
        model_name,
        num_l1,
        num_l2,
        num_l3,
        num_l4,
        num_l5,
        dropout_rate=0.3
    ):
        super(HierarchicalMuRIL, self).__init__()

        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size

        self.dropout = nn.Dropout(dropout_rate)

        self.level1_head = nn.Linear(hidden_size, num_l1)

        self.level2_head = nn.Linear(hidden_size + num_l1, num_l2)

        self.level3_head = nn.Linear(hidden_size + num_l1 + num_l2, num_l3)

        self.level4_head = nn.Linear(hidden_size + num_l1 + num_l2 + num_l3, num_l4)

        self.level5_head = nn.Linear(hidden_size + num_l1 + num_l2 + num_l3 + num_l4, num_l5)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)

        logits_l1 = self.level1_head(cls_output)
        prob_l1 = torch.softmax(logits_l1, dim=1)

        input_l2 = torch.cat([cls_output, prob_l1], dim=1)
        logits_l2 = self.level2_head(input_l2)
        prob_l2 = torch.softmax(logits_l2, dim=1)

        input_l3 = torch.cat([cls_output, prob_l1, prob_l2], dim=1)
        logits_l3 = self.level3_head(input_l3)
        prob_l3 = torch.softmax(logits_l3, dim=1)

        input_l4 = torch.cat([cls_output, prob_l1, prob_l2, prob_l3], dim=1)
        logits_l4 = self.level4_head(input_l4)
        prob_l4 = torch.softmax(logits_l4, dim=1)

        input_l5 = torch.cat([cls_output, prob_l1, prob_l2, prob_l3, prob_l4], dim=1)
        logits_l5 = self.level5_head(input_l5)

        return {
            "level1": logits_l1,
            "level2": logits_l2,
            "level3": logits_l3,
            "level4": logits_l4,
            "level5": logits_l5
        }

In [22]:
model = HierarchicalMuRIL(
    model_name=MODEL_NAME,
    num_l1=num_labels[LEVEL1_COL],
    num_l2=num_labels[LEVEL2_COL],
    num_l3=num_labels[LEVEL3_COL],
    num_l4=num_labels[LEVEL4_COL],
    num_l5=num_labels[LEVEL5_COL],
    dropout_rate=0.3
)

model = model.to(device)

pytorch_model.bin:   0%|          | 0.00/953M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/953M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [23]:
criterion = nn.CrossEntropyLoss()

loss_weights = {
    "level1": 1.0,
    "level2": 1.0,
    "level3": 1.0,
    "level4": 1.0,
    "level5": 1.0
}

In [24]:
EPOCHS = 5
LEARNING_RATE = 2e-5

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

In [25]:
def train_one_epoch(model, dataloader, optimizer, scheduler, criterion, device):
    model.train()

    total_loss = 0

    for batch in dataloader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        labels_l1 = batch["labels"][LEVEL1_COL + "_enc"].to(device)
        labels_l2 = batch["labels"][LEVEL2_COL + "_enc"].to(device)
        labels_l3 = batch["labels"][LEVEL3_COL + "_enc"].to(device)
        labels_l4 = batch["labels"][LEVEL4_COL + "_enc"].to(device)
        labels_l5 = batch["labels"][LEVEL5_COL + "_enc"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        loss_l1 = criterion(outputs["level1"], labels_l1)
        loss_l2 = criterion(outputs["level2"], labels_l2)
        loss_l3 = criterion(outputs["level3"], labels_l3)
        loss_l4 = criterion(outputs["level4"], labels_l4)
        loss_l5 = criterion(outputs["level5"], labels_l5)

        loss = (
            loss_weights["level1"] * loss_l1 +
            loss_weights["level2"] * loss_l2 +
            loss_weights["level3"] * loss_l3 +
            loss_weights["level4"] * loss_l4 +
            loss_weights["level5"] * loss_l5
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    return avg_loss

In [26]:
def evaluate(model, dataloader, criterion, device):
    model.eval()

    total_loss = 0

    true_l1, pred_l1 = [], []
    true_l2, pred_l2 = [], []
    true_l3, pred_l3 = [], []
    true_l4, pred_l4 = [], []
    true_l5, pred_l5 = [], []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            labels_l1 = batch["labels"][LEVEL1_COL + "_enc"].to(device)
            labels_l2 = batch["labels"][LEVEL2_COL + "_enc"].to(device)
            labels_l3 = batch["labels"][LEVEL3_COL + "_enc"].to(device)
            labels_l4 = batch["labels"][LEVEL4_COL + "_enc"].to(device)
            labels_l5 = batch["labels"][LEVEL5_COL + "_enc"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            loss_l1 = criterion(outputs["level1"], labels_l1)
            loss_l2 = criterion(outputs["level2"], labels_l2)
            loss_l3 = criterion(outputs["level3"], labels_l3)
            loss_l4 = criterion(outputs["level4"], labels_l4)
            loss_l5 = criterion(outputs["level5"], labels_l5)

            loss = (
                loss_weights["level1"] * loss_l1 +
                loss_weights["level2"] * loss_l2 +
                loss_weights["level3"] * loss_l3 +
                loss_weights["level4"] * loss_l4 +
                loss_weights["level5"] * loss_l5
            )

            total_loss += loss.item()

            p1 = torch.argmax(outputs["level1"], dim=1)
            p2 = torch.argmax(outputs["level2"], dim=1)
            p3 = torch.argmax(outputs["level3"], dim=1)
            p4 = torch.argmax(outputs["level4"], dim=1)
            p5 = torch.argmax(outputs["level5"], dim=1)

            true_l1.extend(labels_l1.cpu().numpy())
            pred_l1.extend(p1.cpu().numpy())

            true_l2.extend(labels_l2.cpu().numpy())
            pred_l2.extend(p2.cpu().numpy())

            true_l3.extend(labels_l3.cpu().numpy())
            pred_l3.extend(p3.cpu().numpy())

            true_l4.extend(labels_l4.cpu().numpy())
            pred_l4.extend(p4.cpu().numpy())

            true_l5.extend(labels_l5.cpu().numpy())
            pred_l5.extend(p5.cpu().numpy())

    results = {
        "loss": total_loss / len(dataloader),

        "level1_acc": accuracy_score(true_l1, pred_l1),
        "level1_macro_f1": f1_score(true_l1, pred_l1, average="macro"),

        "level2_acc": accuracy_score(true_l2, pred_l2),
        "level2_macro_f1": f1_score(true_l2, pred_l2, average="macro"),

        "level3_acc": accuracy_score(true_l3, pred_l3),
        "level3_macro_f1": f1_score(true_l3, pred_l3, average="macro"),

        "level4_acc": accuracy_score(true_l4, pred_l4),
        "level4_macro_f1": f1_score(true_l4, pred_l4, average="macro"),

        "level5_acc": accuracy_score(true_l5, pred_l5),
        "level5_macro_f1": f1_score(true_l5, pred_l5, average="macro"),
    }

    predictions = {
        "true_l1": true_l1, "pred_l1": pred_l1,
        "true_l2": true_l2, "pred_l2": pred_l2,
        "true_l3": true_l3, "pred_l3": pred_l3,
        "true_l4": true_l4, "pred_l4": pred_l4,
        "true_l5": true_l5, "pred_l5": pred_l5,
    }

    return results, predictions

In [27]:
best_val_f1 = 0
best_model_path = "best_hierarchical_muril.pt"

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    print("-" * 50)

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scheduler,
        criterion,
        device
    )

    val_results, _ = evaluate(
        model,
        val_loader,
        criterion,
        device
    )

    print("Train Loss:", train_loss)
    print("Validation Loss:", val_results["loss"])
    print("Level 1 Macro-F1:", val_results["level1_macro_f1"])
    print("Level 2 Macro-F1:", val_results["level2_macro_f1"])
    print("Level 3 Macro-F1:", val_results["level3_macro_f1"])
    print("Level 4 Macro-F1:", val_results["level4_macro_f1"])
    print("Level 5 Macro-F1:", val_results["level5_macro_f1"])

    avg_macro_f1 = (
        val_results["level1_macro_f1"] +
        val_results["level2_macro_f1"] +
        val_results["level3_macro_f1"] +
        val_results["level4_macro_f1"] +
        val_results["level5_macro_f1"]
    ) / 5

    print("Average Macro-F1:", avg_macro_f1)

    if avg_macro_f1 > best_val_f1:
        best_val_f1 = avg_macro_f1
        torch.save(model.state_dict(), best_model_path)
        print("Best model saved.")


Epoch 1/5
--------------------------------------------------
Train Loss: 9.036540070870766
Validation Loss: 8.636971445644603
Level 1 Macro-F1: 0.4912193505632869
Level 2 Macro-F1: 0.10714285714285715
Level 3 Macro-F1: 0.22493715434891906
Level 4 Macro-F1: 0.02528735632183908
Level 5 Macro-F1: 0.13566492737272598
Average Macro-F1: 0.19685032914992565
Best model saved.

Epoch 2/5
--------------------------------------------------
Train Loss: 8.324262993676323
Validation Loss: 8.017091540729298
Level 1 Macro-F1: 0.525737527401421
Level 2 Macro-F1: 0.19889982686518395
Level 3 Macro-F1: 0.43508479288295804
Level 4 Macro-F1: 0.02528735632183908
Level 5 Macro-F1: 0.13566492737272598
Average Macro-F1: 0.2641348861688256
Best model saved.

Epoch 3/5
--------------------------------------------------
Train Loss: 7.749986044446328
Validation Loss: 7.554870815838084
Level 1 Macro-F1: 0.5185533136424564
Level 2 Macro-F1: 0.1862680155121518
Level 3 Macro-F1: 0.4854752551391573
Level 4 Macro-F1: 0.

In [28]:
model.load_state_dict(torch.load(best_model_path, map_location=device))
model = model.to(device)

In [29]:
test_results, test_predictions = evaluate(
    model,
    test_loader,
    criterion,
    device
)

print(test_results)

{'loss': 7.1218186126035805, 'level1_acc': 0.8120300751879699, 'level1_macro_f1': 0.5418628415803497, 'level2_acc': 0.5939849624060151, 'level2_macro_f1': 0.24606926764843262, 'level3_acc': 0.6578947368421053, 'level3_macro_f1': 0.6018456762353656, 'level4_acc': 0.5714285714285714, 'level4_macro_f1': 0.030303030303030304, 'level5_acc': 0.8984962406015038, 'level5_macro_f1': 0.1352192362093352}


In [32]:
def print_report(level_name, true_values, pred_values, label_col):
    print("\n" + "="*80)
    print(level_name)
    print("="*80)

    target_names = label_encoders[label_col].classes_
    # Explicitly provide the labels to consider, which are the integer representations
    # from 0 up to the number of classes for this level.
    labels_to_report = list(range(num_labels[label_col]))

    print(classification_report(
        true_values,
        pred_values,
        target_names=target_names,
        labels=labels_to_report, # Add this parameter
        zero_division=0
    ))

In [33]:
print_report(
    "Level 1: Hate / Non-Hate",
    test_predictions["true_l1"],
    test_predictions["pred_l1"],
    LEVEL1_COL
)

print_report(
    "Level 2: Target",
    test_predictions["true_l2"],
    test_predictions["pred_l2"],
    LEVEL2_COL
)

print_report(
    "Level 3: Category",
    test_predictions["true_l3"],
    test_predictions["pred_l3"],
    LEVEL3_COL
)

print_report(
    "Level 4: Subcategory",
    test_predictions["true_l4"],
    test_predictions["pred_l4"],
    LEVEL4_COL
)

print_report(
    "Level 5: Bias",
    test_predictions["true_l5"],
    test_predictions["pred_l5"],
    LEVEL5_COL
)


Level 1: Hate / Non-Hate
              precision    recall  f1-score   support

        HATE       0.86      0.77      0.81       279
    NON-HATE       0.77      0.86      0.81       252
         Nil       0.00      0.00      0.00         1

    accuracy                           0.81       532
   macro avg       0.54      0.54      0.54       532
weighted avg       0.82      0.81      0.81       532


Level 2: Target
                     precision    recall  f1-score   support

Hate against LGBTQ+       0.00      0.00      0.00        10
   Hate against Men       0.00      0.00      0.00        74
 Hate against Women       0.47      0.45      0.46       147
    Men- no profane       0.00      0.00      0.00         0
                Nil       0.64      0.97      0.77       257
 No Target- Profane       0.00      0.00      0.00        44

           accuracy                           0.59       532
          macro avg       0.18      0.24      0.21       532
       weighted avg      

In [34]:
def predict_text(text, model, tokenizer, max_len=128):
    model.eval()

    encoding = tokenizer(
        text,
        add_special_tokens=True,
        max_length=max_len,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    pred_l1 = torch.argmax(outputs["level1"], dim=1).cpu().item()
    pred_l2 = torch.argmax(outputs["level2"], dim=1).cpu().item()
    pred_l3 = torch.argmax(outputs["level3"], dim=1).cpu().item()
    pred_l4 = torch.argmax(outputs["level4"], dim=1).cpu().item()
    pred_l5 = torch.argmax(outputs["level5"], dim=1).cpu().item()

    result = {
        "Text": text,
        "Level 1 - Class": label_encoders[LEVEL1_COL].inverse_transform([pred_l1])[0],
        "Level 2 - Target": label_encoders[LEVEL2_COL].inverse_transform([pred_l2])[0],
        "Level 3 - Category": label_encoders[LEVEL3_COL].inverse_transform([pred_l3])[0],
        "Level 4 - Subcategory": label_encoders[LEVEL4_COL].inverse_transform([pred_l4])[0],
        "Level 5 - Bias": label_encoders[LEVEL5_COL].inverse_transform([pred_l5])[0],
    }

    return result

In [35]:
sample_text = "இந்த மாதிரி பெண்களை இழிவாக பேச கூடாது"

predict_text(sample_text, model, tokenizer, MAX_LEN)

{'Text': 'இந்த மாதிரி பெண்களை இழிவாக பேச கூடாது',
 'Level 1 - Class': 'NON-HATE',
 'Level 2 - Target': 'Nil',
 'Level 3 - Category': 'Nil',
 'Level 4 - Subcategory': 'Nil',
 'Level 5 - Bias': 'Nil'}

In [36]:
sample_text = "Oiye Julie vaaya vechikitu summa irundhirukama pechadi pesuney Mannar parambara mannangadi parambara naalu vaartha pesiye votes three digit la iruku"

predict_text(sample_text, model, tokenizer, MAX_LEN)

{'Text': 'Oiye Julie vaaya vechikitu summa irundhirukama pechadi pesuney Mannar parambara mannangadi parambara naalu vaartha pesiye votes three digit la iruku',
 'Level 1 - Class': 'HATE',
 'Level 2 - Target': 'Nil',
 'Level 3 - Category': 'Nil',
 'Level 4 - Subcategory': 'Nil',
 'Level 5 - Bias': 'Nil'}

In [37]:
sample_text = "நான் நினைக்கிறேன் ஜூலிக்கு காசு கொடுத்து மிரட்டி பேச சொல்றாங்க... இதுவும் பயந்துகிட்டு காச வாங்கிட்டு பயந்துகிட்டு எதையாச்சும் பேசலாம்னு வந்திருக்கு..... அதோட புருஷன் என்னதான் பண்றாரு கல்யாணம் ஆயிடுச்சு அடக்கி வைக்க தெரியாதா"

predict_text(sample_text, model, tokenizer, MAX_LEN)

{'Text': 'நான் நினைக்கிறேன் ஜூலிக்கு காசு கொடுத்து மிரட்டி பேச சொல்றாங்க... இதுவும் பயந்துகிட்டு காச வாங்கிட்டு பயந்துகிட்டு எதையாச்சும் பேசலாம்னு வந்திருக்கு..... அதோட புருஷன் என்னதான் பண்றாரு கல்யாணம் ஆயிடுச்சு அடக்கி வைக்க தெரியாதா',
 'Level 1 - Class': 'NON-HATE',
 'Level 2 - Target': 'Nil',
 'Level 3 - Category': 'Nil',
 'Level 4 - Subcategory': 'Nil',
 'Level 5 - Bias': 'Nil'}

In [43]:
sample_text = "உன் வீட்டில் குழந்தை பிறந்தாலும் அதற்கும் காரணம் ஒரு இஸ்லாமியர் தான் என்று நீ சொல்லடா பைத்தியக்கார பயலே"

predict_text(sample_text, model, tokenizer, MAX_LEN)

{'Text': 'உன் வீட்டில் குழந்தை பிறந்தாலும் அதற்கும் காரணம் ஒரு இஸ்லாமியர் தான் என்று நீ சொல்லடா பைத்தியக்கார பயலே',
 'Level 1 - Class': 'HATE',
 'Level 2 - Target': 'Hate against Women',
 'Level 3 - Category': 'IMPLICIT',
 'Level 4 - Subcategory': 'Nil',
 'Level 5 - Bias': 'Nil'}

In [44]:
def apply_consistency_rules(result):
    level1 = result["Level 1 - Class"].lower()

    if "non" in level1 or "no" in level1:
        result["Level 2 - Target"] = "Nil"
        result["Level 3 - Category"] = "Nil"
        result["Level 4 - Subcategory"] = "Nil"
        result["Level 5 - Bias"] = "Nil"

    return result

In [46]:
import pickle

save_dir = "hierarchical_muril_hatespeech"
os.makedirs(save_dir, exist_ok=True)

torch.save(model.state_dict(), os.path.join(save_dir, "model.pt"))
tokenizer.save_pretrained(save_dir)

with open(os.path.join(save_dir, "label_encoders.pkl"), "wb") as f:
    pickle.dump(label_encoders, f)

print("Model saved to:", save_dir)

Model saved to: hierarchical_muril_hatespeech


In [47]:
import pickle
from transformers import AutoTokenizer

save_dir = "hierarchical_muril_hatespeech"

tokenizer = AutoTokenizer.from_pretrained(save_dir)

with open(os.path.join(save_dir, "label_encoders.pkl"), "rb") as f:
    label_encoders = pickle.load(f)

loaded_model = HierarchicalMuRIL(
    model_name=MODEL_NAME,
    num_l1=num_labels[LEVEL1_COL],
    num_l2=num_labels[LEVEL2_COL],
    num_l3=num_labels[LEVEL3_COL],
    num_l4=num_labels[LEVEL4_COL],
    num_l5=num_labels[LEVEL5_COL],
    dropout_rate=0.3
)

loaded_model.load_state_dict(torch.load(os.path.join(save_dir, "model.pt"), map_location=device))
loaded_model = loaded_model.to(device)
loaded_model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HierarchicalMuRIL(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(197285, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwis